In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta
import os
from dotenv import load_dotenv
from fredapi import Fred
import warnings
warnings.filterwarnings('ignore')

### Lite Version

In [295]:
class DCFModel:
    def __init__(self, ticker):
        self.ticker = ticker
        self.financials = None
        self.market_data = None
        self.assumptions = {}  # will hold all user/derived assumptions

    def fetch_financials(self, years=5):
        """Retrieve income statement, balance sheet, cash flow from yfinance."""
        stock = yf.Ticker(self.ticker)
        self.financials = {
            'income': stock.financials.T,
            'balance': stock.balance_sheet.T,
            'cashflow': stock.cashflow.T
        }
        for key in self.financials:
            self.financials[key].index = pd.to_datetime(self.financials[key].index)
            self.financials[key] = self.financials[key].sort_index(ascending=True)

    def fetch_market_data(self):
        """Get current price, shares, beta, risk-free rate, etc."""
        stock = yf.Ticker(self.ticker)
        self.market_data = {
            'price': stock.info.get('currentPrice'),
            'shares_out': stock.info.get('sharesOutstanding'),
            'beta': stock.info.get('beta'),
        }
        # Get 10y treasury from FRED
        self.market_data['rf_rate'] = self.get_risk_free_rate()

    def get_risk_free_rate(self):
        load_dotenv()
        fred = Fred(api_key=os.getenv('FRED_API_KEY'))
        # Get the most recent observation (last available date)
        data = fred.get_series('DGS10')
        # data is a Series with dates as index; take last non-null value
        latest = data.dropna().iloc[-1]
        return latest / 100.0

    def set_assumptions(self, user_assumptions=None):
        """
        Define forecast assumptions.
        user_assumptions: dict with keys like 'revenue_growth', 'op_margin', etc.
        If not provided, derive from historical averages or consensus.
        """
        self.derive_assumptions_from_history()
        if user_assumptions:
            self.assumptions.update(user_assumptions)

    def derive_assumptions_from_history(self):
        """Calculate historical averages for key drivers."""

        rev = self.financials['income']['Total Revenue']
        growth_rates = rev.pct_change().dropna()
        self.assumptions['revenue_growth'] = growth_rates.tail(3).ewm(span=3, adjust=False).mean().iloc[-1]

        ebit = self.financials['income']['EBIT']  # or Operating Income
        op_margin = (ebit / rev).dropna()
        self.assumptions['op_margin'] = op_margin.tail(3).ewm(span=3, adjust=False).mean().iloc[-1]
        
        # Tax rate
        taxes = self.financials['income']['Tax Provision']
        pretax_income = self.financials['income']['Pretax Income']
        tax_rate = (taxes / pretax_income).dropna()
        self.assumptions['tax_rate'] = tax_rate.tail(3).ewm(span=3, adjust=False).mean().iloc[-1]
        
        # D&A as % of revenue
        da = self.financials['cashflow']['Depreciation And Amortization']
        self.assumptions['da_pct_rev'] = (da / rev).dropna().tail(3).ewm(span=3, adjust=False).mean().iloc[-1]
        
        # CapEx as % of revenue
        capex = self.financials['cashflow']['Capital Expenditure'].abs()
        self.assumptions['capex_pct_rev'] = (capex / rev).dropna().tail(3).ewm(span=3, adjust=False).mean().iloc[-1]
        
        # NWC as % of revenue
        self.assumptions['nwc_pct_rev'] = 0.05  # or derive from balance sheet changes

        # Cost of debt
        interest = self.financials['income']['Interest Expense']
        debt = self.financials['balance']['Total Debt']
        combined = pd.concat([interest, debt], axis=1, keys=['interest', 'debt']).dropna()
        cost_debt_series = combined['interest'] / combined['debt']
        self.assumptions["cost_debt"] = cost_debt_series.ewm(span=3, adjust=False).mean().iloc[-1]
        
        # Add default assumptions if not already set
        self.assumptions.setdefault('erp', 0.0423)  # equity risk premium from Damodaran
        self.assumptions.setdefault('terminal_growth', 0.03)

    def calculate_fcff(self, forecast_years=5):
        """Forecast FCFF for each year based on assumptions."""
        # Starting point: latest revenue (TTM)
        last_rev = self.financials['income']['Total Revenue'].iloc[-1]
        fcff_forecast = []
        for year in range(1, forecast_years+1):
            rev_growth = self.assumptions.get(f'rev_growth_y{year}', self.assumptions['revenue_growth'])
            revenue = last_rev * (1 + rev_growth)
            ebit = revenue * self.assumptions['op_margin']
            tax = ebit * self.assumptions['tax_rate']
            d_a = revenue * self.assumptions['da_pct_rev']
            capex = revenue * self.assumptions['capex_pct_rev']
            nwc_change = revenue * self.assumptions['nwc_pct_rev'] - last_rev * self.assumptions['nwc_pct_rev']
            fcff = ebit - tax + d_a - capex - nwc_change
            fcff_forecast.append(fcff)
            last_rev = revenue
        return fcff_forecast

    def calculate_wacc(self):
        """Compute WACC using market data and assumptions."""
        cost_debt = self.assumptions.get('cost_debt', 0.058)
        cost_equity = self.market_data['rf_rate'] + self.market_data['beta'] * (self.assumptions['erp'] + self.assumptions.get("cds", 0))
        # get market values of debt and equity
        market_equity = self.market_data['price'] * self.market_data['shares_out']
        # total debt from latest balance sheet
        total_debt = self.financials['balance']['Total Debt'].iloc[-1]  # may need mapping
        total_capital = market_equity + total_debt
        wacc = (market_equity / total_capital) * cost_equity + \
               (total_debt / total_capital) * cost_debt * (1 - self.assumptions['tax_rate'])
        return wacc

    def terminal_value(self, last_fcff, wacc, sensitivity, method='perpetuity'):
        if method == 'perpetuity':
            if not sensitivity:
                self.assumptions['terminal_growth'] = 0.03
            g = self.assumptions['terminal_growth']
            return last_fcff * (1 + g) / (wacc - g)
        else:  # exit multiple
            multiple = self.assumptions['exit_multiple']
            return last_fcff * multiple  # simplified; usually use EBITDA

    def valuation(self, sensitivity=False):
        """Run the full DCF and return equity value per share."""
        fcff_list = self.calculate_fcff()
        self.assumptions['wacc'] = self.calculate_wacc() if not sensitivity else self.assumptions['wacc']
        wacc = self.assumptions['wacc']
        # discount forecast period
        pv_fcff = sum([f / ((1 + wacc) ** (i+1)) for i, f in enumerate(fcff_list)])
        # terminal value
        tv = self.terminal_value(fcff_list[-1], wacc, sensitivity)
        pv_tv = tv / ((1 + wacc) ** len(fcff_list))
        ev = pv_fcff + pv_tv
        # net debt
        cash = self.financials['balance']['Cash And Cash Equivalents'].iloc[-1]
        debt = self.financials['balance']['Total Debt'].iloc[-1]
        net_debt = debt - cash
        equity_value = ev - net_debt
        shares = self.market_data['shares_out']
        price_target = equity_value / shares
        return price_target

    def sensitivity_analysis(self, wacc_range, g_range):
        """Example: vary WACC and terminal growth."""
        results = pd.DataFrame(index=wacc_range, columns=g_range)
        for w in wacc_range:
            for g in g_range:
                self.assumptions['wacc'] = w  # override
                self.assumptions['terminal_growth'] = g
                results.loc[w, g] = self.valuation(sensitivity=True)
        return results

In [351]:
model = DCFModel('0175.HK')
model.fetch_financials()
model.fetch_market_data()

In [352]:
assumptions = {
    # 'revenue_growth': 0.2,   # 8% for next 5 years
    # 'op_margin': 0.3,
    # 'tax_rate': 0.16,
    # 'capex_pct_rev': 0.03,
    # 'nwc_pct_rev': 0.02,
    # 'terminal_growth': 0.035,
    'erp': 0.05,
}
model.set_assumptions(assumptions)

In [353]:
price = model.valuation()
print(pd.DataFrame([model.assumptions]).T)
print(f"Implied share price: ${price:.2f}")
model.sensitivity_analysis(np.arange(0.08, 0.11, 0.005), np.arange(0.03, 0.04, 0.005))

                        0
revenue_growth   0.336999
op_margin        0.055621
tax_rate         0.046068
da_pct_rev       0.045052
capex_pct_rev    0.066557
nwc_pct_rev      0.050000
cost_debt        0.066792
erp              0.050000
terminal_growth  0.030000
wacc             0.081627
Implied share price: $31.65


,0.030,0.035,0.040
0.080,32.647334,35.591969,39.272762
0.085,29.759533,32.124676,35.015405
0.090,27.356707,29.291743,31.613786
0.095,25.326856,26.934493,28.834427
0.100,23.589981,24.943072,26.521677
0.105,22.08742,23.239017,24.567783


### Todo
Segment‑level forecasting if the company reports segments.

Market Value for debt, Preferred Stocks, Non Controlling Interest

Debt schedules and cash sweep for companies with complex capital structures.

Pension and other post‑employment liabilities adjustments.

Monte Carlo simulation for probabilistic valuation

### Sophisticated Version

In [ ]:
class EnhancedDCFModel:
    """
    Professional DCF Valuation Model with enhanced data handling
    and sophisticated assumption derivation.
    """
    
    def __init__(self, ticker, fred_api_key=None):
        """
        Initialize DCF model for a given ticker.
        
        Parameters:
        -----------
        ticker : str
            Stock ticker symbol
        fred_api_key : str, optional
            FRED API key for risk-free rate
        """
        self.ticker = ticker.upper()
        self.fred_api_key = fred_api_key or os.getenv('FRED_API_KEY')
        
        # Data containers
        self.financials = {}
        self.mapped = {}
        self.market_data = {}
        self.assumptions = {}
        self.earnings_quality = None

        self.fetch_all_data()

        # Initalize assumptions
        self.derive_assumptions_from_history(forecast_years=5)
        
    def fetch_all_data(self):
        # Fetch financial statements
        self.fetch_financials()

        # Fetch market data
        self.fetch_market_data()
        
        # Analyze earnings quality
        self.analyze_earnings_quality()
        
    def fetch_financials(self):
        """Fetch and standardize financial statements from yfinance."""
        stock = yf.Ticker(self.ticker)
        
        # Fetch statements and transpose so dates are index
        self.financials = {
            'income': stock.income_stmt.T,
            'balance': stock.balance_sheet.T,
            'cashflow': stock.cashflow.T
        }
        
        # Check if data is available
        if self.financials['income'].empty:
            raise ValueError(f"No income statement data found for {self.ticker}")
        
        # Create standardized mapping
        self._create_standardized_mapping()
        
        # Sort all dataframes descending (most recent first)
        for key in self.financials:
            self.financials[key] = self.financials[key].sort_index(ascending=False)
    
    def _create_standardized_mapping(self):
        """Map yfinance columns to standardized names for DCF."""
        inc = self.financials['income']
        cf = self.financials['cashflow']
        
        # Helper function for safe column access
        def safe_get(df, possible_names):
            for name in possible_names:
                if name in df.columns:
                    return df[name]
            return pd.Series(index=df.index, dtype=float)
        
        # === Income Statement Mapping ===
        self.mapped['revenue'] = safe_get(inc, 
            ['Total Revenue', 'Operating Revenue'])
        
        # Operating Income / EBIT (prioritized)
        self.mapped['ebit'] = safe_get(inc, 
            ['EBIT', 'Operating Income', 'Total Operating Income As Reported'])
        
        # EBITDA (prefer normalized if available)
        self.mapped['ebitda'] = safe_get(inc, 
            ['Normalized EBITDA', 'EBITDA'])
        
        # Normalized metrics (exclude one-time items)
        self.mapped['normalized_ebitda'] = safe_get(inc, ['Normalized EBITDA'])
        self.mapped['normalized_income'] = safe_get(inc, ['Normalized Income'])
        
        # Tax-related
        self.mapped['tax_provision'] = safe_get(inc, ['Tax Provision'])
        self.mapped['pretax_income'] = safe_get(inc, ['Pretax Income'])
        self.mapped['tax_rate_calc'] = safe_get(inc, ['Tax Rate For Calcs'])
        
        # Depreciation & Amortization (from income statement)
        self.mapped['da_income'] = safe_get(inc, 
            ['Reconciled Depreciation', 'Depreciation And Amortization In Income Statement',
             'Depreciation Income Statement'])
        
        # Interest
        self.mapped['interest_expense'] = safe_get(inc, 
            ['Interest Expense', 'Interest Expense Non Operating'])
        self.mapped['interest_income'] = safe_get(inc, 
            ['Interest Income', 'Interest Income Non Operating'])
        
        # EPS and shares
        self.mapped['diluted_shares'] = safe_get(inc, ['Diluted Average Shares'])
        self.mapped['basic_shares'] = safe_get(inc, ['Basic Average Shares'])
        self.mapped['diluted_eps'] = safe_get(inc, ['Diluted EPS'])
        self.mapped['basic_eps'] = safe_get(inc, ['Basic EPS'])
        
        # Unusual items
        self.mapped['unusual_items'] = safe_get(inc, 
            ['Total Unusual Items', 'Special Income Charges']).fillna(0)
        
        # === Cash Flow Statement Mapping ===
        self.mapped['capex'] = safe_get(cf, 
            ['Capital Expenditure', 'Capital Expenditures'])
        self.mapped['da_cf'] = safe_get(cf, 
            ['Depreciation And Amortization', 'Depreciation & Amortization'])
        self.mapped['change_in_working_capital'] = safe_get(cf, 
            ['Changes In Working Capital', 'Change In Working Capital'])
        
        # === Ensure all series are numeric and handle NaN ===
        for key in self.mapped:
            if not self.mapped[key].empty:
                self.mapped[key] = pd.to_numeric(self.mapped[key], errors='coerce')
        
        # Use D&A from cash flow if income statement version is missing
        if self.mapped['da_income'].empty and not self.mapped['da_cf'].empty:
            self.mapped['da_income'] = self.mapped['da_cf']
    
    def fetch_market_data(self):
        """Fetch market data including risk-free rate, beta, etc."""
        stock = yf.Ticker(self.ticker)
        info = stock.info
        
        # Basic market data
        self.market_data['price'] = info.get('currentPrice', info.get('regularMarketPrice'))
        self.market_data['shares_out'] = info.get('sharesOutstanding')
        self.market_data['beta'] = info.get('beta', 1.0)  # Default to 1 if missing
        self.market_data['market_cap'] = info.get('marketCap')
        
        # Get risk-free rate (10-year Treasury)
        self.market_data['rf_rate'] = self._get_risk_free_rate()
        
        # Get equity risk premium (Damodaran's data - using conservative default)
        self.market_data['erp'] = 0.0423  # 4.5% is a reasonable long-term average
        
        # Industry/sector
        self.market_data['sector'] = info.get('sector')
        self.market_data['industry'] = info.get('industry')
    
    def _get_risk_free_rate(self, method='fredapi'):        
        # Try fredapi first
        if method == 'fredapi' and self.fred_api_key:
            try:
                fred = Fred(api_key=self.fred_api_key)
                data = fred.get_series('DGS10')
                latest = data.dropna().iloc[-1]
                return latest / 100.0
            except Exception as e:
                print(f"fredapi failed: {e}")
        
        # Fallback
        print("WARNING:Using default 4.0% risk-free rate")
        return 0.04
    
    def analyze_earnings_quality(self):
        """
        Analyze earnings quality based on unusual items.
        Returns recommendation for using normalized figures.
        """
        if self.mapped['unusual_items'].empty or self.mapped['revenue'].empty:
            self.earnings_quality = 'unknown'
            return
        
        # Calculate unusual items as % of revenue
        unusual_pct = (self.mapped['unusual_items'].abs() / self.mapped['revenue']).mean()
        
        if unusual_pct > 0.10:
            self.earnings_quality = 'poor'
            print(f"Poor earnings quality: {unusual_pct:.1%} unusual items")
            print("   → Recommend using normalized figures")
        elif unusual_pct > 0.03:
            self.earnings_quality = 'moderate'
            print(f"Moderate earnings quality: {unusual_pct:.1%} unusual items")
        else:
            self.earnings_quality = 'good'
            print(f"Good earnings quality: {unusual_pct:.1%} unusual items")
    
    def calculate_historical_cost_of_debt(self, method='ema', ema_span=3):

        # Get interest expense
        interest = self.mapped['interest_expense']
        
        # Get total debt from balance sheet
        debt = self.financials['balance']['Total Debt']
        
        # Ensure aligned data
        combined = pd.concat([interest, debt], axis=1, keys=['interest', 'debt']).dropna()
        
        if combined.empty or (combined['debt'].abs() < 1e6).all():
            print("WARNING: Insufficient debt data, using synthetic rate")
            return self.market_data['rf_rate'] + 0.02  # Risk-free + 2% spread
        
        # Calculate average debt (use current and prior year)
        debt_avg = (combined['debt'] + combined['debt'].shift(1)) / 2
        combined = pd.concat([combined['interest'], debt_avg], axis=1, keys=['interest', 'debt_avg']).dropna()
        
        # Calculate implied rates
        cost_debt_series = combined['interest'] / combined['debt_avg']
        cost_debt_series = cost_debt_series.replace([np.inf, -np.inf], np.nan).dropna()
        
        if cost_debt_series.empty:
            return self.market_data['rf_rate'] + 0.02
        
        # Apply chosen method
        if method == 'latest':
            return cost_debt_series.iloc[-1]
        elif method == 'average':
            return cost_debt_series.mean()
        else:  # EMA
            return cost_debt_series.ewm(span=ema_span, adjust=False).mean().iloc[-1]
    
    def derive_assumptions_from_history(self, ema_span=3, forecast_years=5):
        """
        Derive forecast assumptions using EMA of historical data.
        Uses normalized figures if earnings quality is poor.
        """
        
        # Choose which metrics to use based on earnings quality
        use_normalized = self.earnings_quality in ['poor', 'moderate']
        
        # === Revenue Growth ===
        revenue = self.mapped['revenue'].dropna()[::-1]
        growth_rates = revenue.pct_change().dropna()
        
        if len(growth_rates) >= 2:
            self.assumptions['revenue_growth'] = growth_rates.ewm(span=ema_span).mean().iloc[-1]
        else:
            self.assumptions['revenue_growth'] = 0.05  # Default 5%
        
        # === Operating Margin ===
        if use_normalized and not self.mapped['normalized_ebitda'].empty:
            op_income = self.mapped['normalized_ebitda']
            margin_name = "Normalized EBITDA margin"
        else:
            op_income = self.mapped['ebit']
            margin_name = "EBIT margin"
        
        # Align with revenue
        aligned = pd.concat([op_income, revenue], axis=1, keys=['op_income', 'revenue']).dropna()
        if not aligned.empty:
            margins = aligned['op_income'] / aligned['revenue']
            margins = margins.replace([np.inf, -np.inf], np.nan).dropna()
            
            if len(margins) >= 2:
                self.assumptions['op_margin'] = margins.ewm(span=ema_span).mean().iloc[-1]
            else:
                self.assumptions['op_margin'] = 0.15  # Default
        else:
            self.assumptions['op_margin'] = 0.15
        
        # === Tax Rate ===
        if not self.mapped['tax_rate_calc'].empty:
            tax_rates = self.mapped['tax_rate_calc']
            tax_rates = tax_rates.replace([np.inf, -np.inf], np.nan).dropna()
            if len(tax_rates) >= 2:
                self.assumptions['tax_rate'] = tax_rates.ewm(span=ema_span).mean().iloc[-1]
            else:
                self.assumptions['tax_rate'] = 0.21  # US statutory
        else:
            # Calculate from provision/pretax
            tax_calc = self.mapped['tax_provision'] / self.mapped['pretax_income']
            tax_calc = tax_calc.replace([np.inf, -np.inf], np.nan).dropna()
            if len(tax_calc) >= 2:
                self.assumptions['tax_rate'] = tax_calc.ewm(span=ema_span).mean().iloc[-1]
            else:
                self.assumptions['tax_rate'] = 0.21
        
        # === Depreciation % of Revenue ===
        da = self.mapped['da_income']
        if not da.empty and not revenue.empty:
            aligned = pd.concat([da, revenue], axis=1, keys=['da', 'revenue']).dropna()
            if not aligned.empty:
                da_pct = aligned['da'] / aligned['revenue']
                da_pct = da_pct.replace([np.inf, -np.inf], np.nan).dropna()
                if len(da_pct) >= 2:
                    self.assumptions['da_pct_rev'] = da_pct.ewm(span=ema_span).mean().iloc[-1]
                else:
                    self.assumptions['da_pct_rev'] = 0.03
            else:
                self.assumptions['da_pct_rev'] = 0.03
        else:
            self.assumptions['da_pct_rev'] = 0.03
        
        # === Capex % of Revenue ===
        capex = self.mapped['capex'].abs()  # Make positive
        if not capex.empty and not revenue.empty:
            aligned = pd.concat([capex, revenue], axis=1, keys=['capex', 'revenue']).dropna()
            if not aligned.empty:
                capex_pct = aligned['capex'] / aligned['revenue']
                capex_pct = capex_pct.replace([np.inf, -np.inf], np.nan).dropna()
                if len(capex_pct) >= 2:
                    self.assumptions['capex_pct_rev'] = capex_pct.ewm(span=ema_span).mean().iloc[-1]
                else:
                    self.assumptions['capex_pct_rev'] = 0.04
            else:
                self.assumptions['capex_pct_rev'] = 0.04
        else:
            self.assumptions['capex_pct_rev'] = 0.04
        
        # === Working Capital % of Revenue (simplified) ===
        # This is a simplification - real models use balance sheet items
        self.assumptions['nwc_pct_rev'] = 0.05  # Default 5%
        
        # === Terminal Growth ===
        self.assumptions['terminal_growth'] = 0.025  # 2.5% long-term
        
        # === Cost of Debt ===
        self.assumptions['cost_debt'] = self.calculate_historical_cost_of_debt(method='ema')
    
    def calculate_wacc(self):
        # Cost of equity: CAPM
        cost_equity = self.market_data['rf_rate'] + self.market_data['beta'] * self.market_data['erp']
        
        # Market values
        market_equity = self.market_data['price'] * self.market_data['shares_out']
        
        # Latest debt from balance sheet (hard to find Market Values)
        total_debt = self.financials['balance']['Total Debt'].iloc[0]
        
        total_capital = market_equity + total_debt

        wacc = (market_equity / total_capital) * cost_equity + \
               (total_debt / total_capital) * self.assumptions['cost_debt'] * (1 - self.assumptions['tax_rate'])
        
        return wacc
    
    def calculate_fcff(self, forecast_years=5):
        """
        Forecast Free Cash Flow to Firm for specified years.
        """
        # Get latest revenue as base
        if self.mapped['revenue'].empty:
            raise ValueError("No revenue data available")
        
        last_revenue = self.mapped['revenue'].iloc[0]
        fcff_list = []
        revenue_list = []
        
        for year in range(1, forecast_years + 1):
            # Revenue with growth
            if year == 1:
                growth = self.assumptions['revenue_growth']
            else:
                # Gradually fade to terminal growth by year 5
                fade = min(1.0, year / forecast_years)
                growth = self.assumptions['revenue_growth'] * (1 - fade) + \
                        self.assumptions['terminal_growth'] * fade
            
            revenue = last_revenue * (1 + growth)
            revenue_list.append(revenue)
            
            ebit = revenue * self.assumptions['op_margin']
            taxes = ebit * self.assumptions['tax_rate']
            da = revenue * self.assumptions['da_pct_rev']
            capex = revenue * self.assumptions['capex_pct_rev']
            nwc_change = revenue * self.assumptions['nwc_pct_rev'] - \
                        last_revenue * self.assumptions['nwc_pct_rev']
            
            fcff = ebit - taxes + da - capex - nwc_change
            fcff_list.append(fcff)
            
            last_revenue = revenue
        
        return fcff_list
    
    def terminal_value(self, last_fcff, wacc, method='perpetuity'):
        """
        Calculate terminal value using either perpetuity growth or exit multiple.
        """
        if method == 'perpetuity':
            g = self.assumptions['terminal_growth']
            if wacc <= g:
                print("ERROR: WACC <= terminal growth, using exit multiple method")
                return self.terminal_value(last_fcff, wacc, method='exit_multiple')
            
            tv = last_fcff * (1 + g) / (wacc - g)
            print(f"Terminal value (perpetuity): ${tv/1e6:.0f}M")
            return tv
        
        else:  # exit multiple - use EBITDA if available
            if not self.mapped['normalized_ebitda'].empty:
                ebitda = self.mapped['normalized_ebitda'].iloc[0]
            elif not self.mapped['ebitda'].empty:
                ebitda = self.mapped['ebitda'].iloc[0]
            else:
                # Estimate EBITDA from EBIT + D&A
                ebit = self.mapped['ebit'].iloc[0]
                da = self.mapped['da_income'].iloc[0] if not self.mapped['da_income'].empty else 0
                ebitda = ebit + da
            
            multiple = self.assumptions.get('ebitda_multiple', 8.0)  # Default 8x
            tv = ebitda * multiple
            print(f"   Terminal value (EBITDA {multiple}x): ${tv/1e6:.0f}M")
            return tv
    
    def valuation(self, forecast_years=5, terminal_method='perpetuity'):
        """
        Perform full DCF valuation.
        """
        
        # Calculate WACC
        wacc = self.calculate_wacc()
        self.assumptions['wacc'] = wacc
        
        # Forecast FCFF
        fcff_list = self.calculate_fcff(forecast_years)
        
        # Calculate present value of forecast period
        pv_fcff = 0
        for i, fcff in enumerate(fcff_list, 1):
            pv_fcff += fcff / ((1 + wacc) ** i)
        
        # Terminal value
        tv = self.terminal_value(fcff_list[-1], wacc, method=terminal_method)
        pv_tv = tv / ((1 + wacc) ** forecast_years)
        
        # Enterprise value
        ev = pv_fcff + pv_tv
        
        # Subtract net debt
        # Get latest cash and debt
        cash = self.financials['balance'].get('Cash And Cash Equivalents', 
                                              self.financials['balance'].get('Cash', pd.Series([0]))).iloc[0]        
        debt = self.financials['balance']['Total Debt'].iloc[0]
       
        net_debt = debt - cash
        equity_value = ev - net_debt
        
        # Per share value
        shares = self.market_data['shares_out']
        
        price_target = equity_value / shares
        print(f"Implied Share Price: ${price_target:.2f}")
        
        # Compare to current price
        current_price = self.market_data['price']
        if current_price:
            diff = (price_target - current_price) / current_price
            print(f"Current Price: ${current_price:.2f}")
            print(f"Upside/Downside: {diff*100:+.1f}%")
        
        return {
            'enterprise_value': ev,
            'equity_value': equity_value,
            'price_target': price_target,
            'wacc': wacc,
            'assumptions': self.assumptions
        }
    
    def sensitivity_analysis(self, wacc_range=None, growth_range=None):
        """
        Perform sensitivity analysis on WACC and terminal growth.
        """
        
        if wacc_range is None:
            wacc_range = np.arange((self.assumptions.get('wacc', 0.08) - 0.02),
                                  self.assumptions.get('wacc', 0.08) + 0.03, 0.005)
        
        if growth_range is None:
            growth_range = np.arange(0.01, 0.04, 0.005)
        
        print(f"\n{'='*60}")
        print("📊 SENSITIVITY ANALYSIS - Price Target by WACC and Terminal Growth")
        print(f"{'='*60}")
        
        results = pd.DataFrame(index=[f"{g*100:.1f}%" for g in growth_range],
                              columns=[f"{w*100:.1f}%" for w in wacc_range])
        
        # Store original assumptions
        original_wacc = self.assumptions.get('wacc')
        original_growth = self.assumptions.get('terminal_growth')
        
        for w in wacc_range:
            for g in growth_range:
                self.assumptions['wacc'] = w
                self.assumptions['terminal_growth'] = g
                
                # Quick valuation (reuse existing forecasts)
                fcff_list = self.calculate_fcff(forecast_years=5)
                pv_fcff = sum([f / ((1 + w) ** (i+1)) for i, f in enumerate(fcff_list)])
                tv = fcff_list[-1] * (1 + g) / (w - g) if w > g else fcff_list[-1] * 10
                pv_tv = tv / ((1 + w) ** 5)
                ev = pv_fcff + pv_tv
                
                # Net debt and shares
                cash = self.financials['balance'].get('Cash And Cash Equivalents', 
                                                      pd.Series([0])).iloc[0]
                debt = self.financials['balance'].get('Total Debt', pd.Series([0])).iloc[0]
                net_debt = debt - cash
                equity_value = ev - net_debt
                
                shares = self.market_data['shares_out']
                results.loc[f"{g*100:.1f}%", f"{w*100:.1f}%"] = equity_value / shares
        
        # Restore original assumptions
        if original_wacc:
            self.assumptions['wacc'] = original_wacc
        if original_growth:
            self.assumptions['terminal_growth'] = original_growth
        
        return results
    
    def generate_report(self):
        """
        Generate a comprehensive valuation report.
        """
        valuation = self.valuation()
        
        print(f"\n{'='*60}")
        print(f"DCF VALUATION REPORT: {self.ticker}")
        print(f"{'='*60}")
        
        print("\nCOMPANY INFORMATION")
        print(f"   Ticker: {self.ticker}")
        print(f"   Sector: {self.market_data.get('sector', 'N/A')}")
        print(f"   Industry: {self.market_data.get('industry', 'N/A')}")
        print(f"   Earnings Quality: {self.earnings_quality}")
        
        print("\nKEY ASSUMPTIONS")
        for key, value in self.assumptions.items():
            if isinstance(value, float):
                print(f"   {key}: {value*100:.2f}%")
        
        print("\nVALUATION SUMMARY")
        print(f"   Enterprise Value: ${valuation['enterprise_value']/1e6:.2f}M")
        print(f"   Equity Value: ${valuation['equity_value']/1e6:.2f}M")
        print(f"   Implied Share Price: ${valuation['price_target']:.2f}")
        print(f"   Current Price: ${self.market_data['price']:.2f}")
        
        if self.market_data['price']:
            diff = (valuation['price_target'] - self.market_data['price']) / self.market_data['price']
            print(f"   Upside/Downside: {diff*100:+.1f}%")
        
        return valuation

    def reset_assumptions(self):
        self.derive_assumptions_from_history(forecast_years=5)

    def set_assumptions(self, user_assumptions):
        self.assumptions.update(user_assumptions)

In [395]:
 # Initialize model for Apple
enhanced_model = EnhancedDCFModel('GOOGL')

Good earnings quality: 2.5% unusual items


In [396]:
assumptions = {
    # 'revenue_growth': 0.2,   # 8% for next 5 years
    # # 'op_margin': 0.3,
    # 'tax_rate': 0.16,
    # # 'capex_pct_rev': 0.03,
    # # 'nwc_pct_rev': 0.02,
    # # 'terminal_growth': 0.035,
    # 'erp': 0.05,
}
enhanced_model.set_assumptions(assumptions)

# # Run valuation
# result = model.valuation(forecast_years=5, terminal_method='perpetuity')

# # Generate full report
result = enhanced_model.generate_report()

# Run sensitivity analysis
sensitivity = enhanced_model.sensitivity_analysis()
print("\nSensitivity Analysis Matrix:")
print(sensitivity)

Terminal value (perpetuity): $1606002M
Implied Share Price: $235.37
Current Price: $307.32
Upside/Downside: -23.4%

DCF VALUATION REPORT: GOOGL

COMPANY INFORMATION
   Ticker: GOOGL
   Sector: Communication Services
   Industry: Internet Content & Information
   Earnings Quality: good

KEY ASSUMPTIONS
   revenue_growth: 13.83%
   op_margin: 35.69%
   tax_rate: 15.49%
   da_pct_rev: 4.80%
   capex_pct_rev: 18.25%
   nwc_pct_rev: 5.00%
   terminal_growth: 2.50%
   cost_debt: 1.10%
   wacc: 8.60%

VALUATION SUMMARY
   Enterprise Value: $1398895.03M
   Equity Value: $1370312.03M
   Implied Share Price: $235.37
   Current Price: $307.32
   Upside/Downside: -23.4%

📊 SENSITIVITY ANALYSIS - Price Target by WACC and Terminal Growth

Sensitivity Analysis Matrix:
            6.6%        7.1%        7.6%        8.1%        8.6%        9.1%  \
1.0%  262.682788   240.43468  221.561974  205.351438  191.277546  178.944536   
1.5%  287.211717  260.799525  238.721926  219.993649  203.906978   189.94045